# Trabajo Práctico - Diseño de Solución de Datos
## Sistema de Agricultura de Precisión basado en IoT y LoRaWAN

---

# 1. Análisis del caso de uso

## 1.1 Descripción del caso de uso
El proyecto consiste en el diseño de una solución de datos para un sistema de Agricultura de Precisión basado en dispositivos IoT y comunicaciones LoRaWAN. El sistema permite monitorear establecimientos agrícolas mediante dispositivos instalados en el campo, almacenar las mediciones generadas y brindar información para el monitoreo, la gestión del riego y futuras aplicaciones de Inteligencia Artificial.

## 1.2 Problema que busca resolver
Los dispositivos instalados en el campo generan información de forma continua y distribuida. La solución propuesta busca centralizar estos datos, administrar la infraestructura agrícola y conservar el historial de mediciones para facilitar el monitoreo, el análisis y la toma de decisiones.


## 1.3 Usuarios principales
El sistema contempla tres perfiles de usuario:
- **Operador:** consulta dispositivos, mediciones, gráficos y alarmas.
- **Configurador:** además de las funciones del Operador, administra configuraciones y alarmas.
- **Administrador:** administra usuarios, perfiles y permisos, además de todas las funciones anteriores.


## 1.4 Procesos y funcionalidades
El sistema deberá permitir:
- Administrar establecimientos, lotes, sectores y pivotes.
- Registrar y administrar dispositivos.
- Gestionar el historial de instalación de los dispositivos.
- Almacenar mediciones y datos de comunicación.
- Monitorear variables en tiempo real.
- Consultar información histórica.
- Administrar alarmas.
- Gestionar usuarios y permisos.


## 1.5 Información que gestiona el sistema
La solución administra información correspondiente a:
- Establecimientos agrícolas.
- Lotes, sectores y pivotes.
- Dispositivos IoT.
- Historial de instalaciones.
- Mediciones.
- Gateways y datos de comunicación.
- Usuarios y perfiles.
- Configuraciones y alarmas.

## 1.6 Riesgos relacionados con los datos
Los principales riesgos considerados son:
- Pérdida de mediciones.
- Inconsistencias en la ubicación de los dispositivos.
- Accesos no autorizados.
- Modificaciones indebidas de configuraciones.
- Crecimiento del volumen de datos.
- Pérdida de integridad de la información.


## 1.7 Principales decisiones de diseño
El diseño de la solución priorizará la integridad, trazabilidad y escalabilidad de los datos. Para ello, se considerará inicialmente utilizar una base de datos relacional PostgreSQL como plataforma principal, complementada con la extensión TimescaleDB para el almacenamiento eficiente de series temporales. Además, las variables medidas por los dispositivos se almacenarán en formato JSONB, permitiendo representar distintos tipos de mediciones de manera flexible.

---

# 2. Relevamiento de datos necesarios
La solución a desarrollar deberá almacenar o consultar los siguientes datos, clasificados según las categorías propuestas:

## 2.1 Datos estructurados
Se identifican los siguientes:
- Campos.
- Lotes.
- Sectores.
- Pivotes.
- Categorías de dispositivos.
- Tipos de dispositivos.
- Dispositivos.
- Instalaciones.
- Gateways.
- Usuarios.
- Perfiles.
- Alarmas.
- Configuraciones.

## 2.2 Datos semiestructurados
Las “mediciones”, ya que las variables medidas difieren según el tipo de dispositivo. Por ejemplo:
1. Estaciones meteorológicas:
    - temperatura.
    - humedad relativa.
    - velocidad y dirección del viento.
    - radiación solar.
    - precipitaciones.
2. sondas de suelo: por cada nivel (6 niveles):
    - humedad.
    - temperatura.
    - conductividad eléctrica.

Las variables medidas se almacenarán en formato JSONB, permitiendo registrar distintos conjuntos de valores sin modificar la estructura de la tabla de mediciones.


## 2.3 Datos no estructurados
No están contemplados por ahora.

## 2.4 Datos operacionales
Consideramos como datos operacionales a aquellos utilizados por la aplicación durante su funcionamiento diario, necesarios para realizar las tareas de monitoreo y administración.
Entre ellos se encuentran:
- Ubicación de sensores.
- Mediciones.
- Alarmas.
- Configuraciones.

## 2.5 Datos analíticos
En esta categoría básicamente se encuentra el histórico de mediciones, utilizado para realizar análisis y apoyar la toma de decisiones.


## 2.6 Datos sensibles
El sistema administra información que requiere protección, entre ella:
- Credenciales de acceso.
- Datos personales de los usuarios.
- Configuraciones del sistema.

## 2.7 Datos de auditoría y trazabilidad
El sistema conservará información que permita reconstruir eventos y realizar auditorías, incluyendo:
- Historial de instalación de los dispositivos.
- Historial de asignación de pivotes a lotes.

(Estos datos permitirán relacionar, por ejemplo, la influencia del riego en las variables medidas en suelo).

- Registros de recepción de mediciones.
- Fecha y hora de cada medición.

(Para corroborar que no se hayan perdido mensajes).
- Cambios realizados sobre configuraciones y alarmas.


## 2.8 Ejemplos de datos

---

# 4. Modelo conceptual

El modelo conceptual fija las entidades del dominio, sus atributos, las relaciones entre ellas, sus cardinalidades y las restricciones que deben cumplirse, con independencia de la tecnología con la que se implemente la solución.

Se construyó a partir del relevamiento del dominio real documentado en `docs/DetallesParaModelado.ipynb`, que recoge cómo opera efectivamente un establecimiento de agricultura de precisión: cómo se subdivide la superficie, cómo se instalan y reubican los dispositivos, cómo se transmiten las mediciones por LoRaWAN y cómo se definen las alarmas.

## 4.1 Alcance del modelo

El modelo conceptual describe **qué** información existe en el dominio y cómo se vincula, no **cómo** se almacena. En consecuencia, no incluye claves foráneas, tipos de datos del motor, índices ni tablas intermedias: todo eso corresponde al modelo de implementación (sección 5) y al modelo físico (sección 8).

Esa decisión tiene una consecuencia visible que conviene anticipar: el modelo conceptual tiene **16 entidades**, mientras que el esquema físico tiene **18 tablas**. La diferencia son las dos tablas asociativas que resuelven las relaciones muchos a muchos, que en este nivel se representan como relaciones y no como entidades (ver sección 4.6).

Sí se conserva la nomenclatura `snake_case` del esquema físico para nombrar entidades y atributos. Es una elección deliberada: aunque un modelo conceptual admitiría nombres de negocio en prosa, mantener un vocabulario único a lo largo del conceptual, el lógico, el físico y el DDL permite leer los cuatro artefactos en paralelo sin traducir nombres en cada salto.

El diagrama se mantiene como fuente Mermaid versionada en `docs/modelo_conceptual.mmd` y se exporta a `docs/modelo_conceptual.png`. Versionar el fuente además de la imagen permite revisar los cambios del modelo en el control de versiones, en lugar de comparar imágenes.

## 4.2 Diagrama entidad-relación

![Modelo conceptual del dominio](modelo_conceptual.png)

El diagrama usa notación de pata de gallo (*crow's foot*): el símbolo del extremo de cada línea indica la cardinalidad de ese lado. Una barra doble representa "exactamente uno", un círculo seguido de pata de gallo representa "cero o muchos". Las relaciones muchos a muchos aparecen con pata de gallo en ambos extremos.

## 4.3 Entidades principales

El dominio se organiza en cinco bloques temáticos, que son los mismos que estructuran el esquema físico (sección 8.1).

**Organización del establecimiento**

| Entidad | Descripción | Atributos relevantes |
| --- | --- | --- |
| `campo` | Establecimiento agrícola | `nombre`, `ubicacion`, `superficie` |
| `lote` | Parcela dentro de un campo | `nombre`, `superficie` |
| `sector` | Subdivisión de un lote | `nombre` |
| `pivote` | Máquina de riego por pivote central | `nombre`, `fabricante`, `modelo` |
| `asignacion_pivote` | Historial de qué pivote riega qué lote | `fecha_inicio`, `fecha_fin` |

**Dispositivos IoT**

| Entidad | Descripción | Atributos relevantes |
| --- | --- | --- |
| `categoria_dispositivo` | Agrupamiento mayor (suelo, riego, meteorológico) | `nombre` |
| `tipo_dispositivo` | Clase de equipo dentro de una categoría | `nombre` |
| `variable` | Magnitud física que un tipo de dispositivo sensa | `nombre`, `unidad` |
| `dispositivo` | Equipo físico instalado en el campo | `numero_serie`, credenciales LoRaWAN (`dev_eui`, `app_eui`, `app_key`), `intervalo_transmision`, `estado_operativo`, `estado_comunicacion` |
| `instalacion_dispositivo` | Historial de dónde estuvo instalado cada dispositivo | `fecha_inicio`, `fecha_fin` |

**Mediciones**

| Entidad | Descripción | Atributos relevantes |
| --- | --- | --- |
| `gateway` | Receptor LoRaWAN que retransmite las mediciones | `nombre` |
| `medicion` | Lectura individual enviada por un dispositivo | `fecha_hora`, `valores_medidos`, `rssi`, `snr`, `contador_mensajes` |

**Alarmas**

| Entidad | Descripción | Atributos relevantes |
| --- | --- | --- |
| `regla_alarma` | Condición que define cuándo generar una alarma | `descripcion`, `umbral_inferior`, `umbral_superior`, `habilitada` |
| `evento_alarma` | Ocurrencia concreta en que una regla se cumplió | `fecha_hora`, `valor_detectado` |

**Usuarios**

| Entidad | Descripción | Atributos relevantes |
| --- | --- | --- |
| `perfil` | Perfil de acceso (Operador, Configurador, Administrador) | `nombre`, `descripcion` |
| `usuario` | Persona que opera el sistema | `nombre`, `apellido`, `email`, `contrasena` |

Los atributos `rssi`, `snr` y `contador_mensajes` de `medicion` no describen la magnitud sensada sino la calidad del enlace de radio y la continuidad de la numeración de mensajes. Se incorporan como atributos de primer orden porque el caso requiere detectar pérdida de transmisiones (sección 2.7), lo que exige poder consultarlos y agregarlos igual que cualquier otra medición.

## 4.4 Relaciones y cardinalidades

| Relación | Cardinalidad | Lectura |
| --- | --- | --- |
| `campo` — `lote` | 1 : N | Un campo posee varios lotes; cada lote pertenece a un solo campo |
| `campo` — `pivote` | 1 : N | Un campo posee varios pivotes; cada pivote pertenece a un solo campo |
| `lote` — `sector` | 1 : N | Un lote se divide en varios sectores |
| `pivote` — `asignacion_pivote` | 1 : N | Un pivote acumula varias asignaciones a lo largo del tiempo |
| `lote` — `asignacion_pivote` | 1 : N | Un lote acumula varias asignaciones a lo largo del tiempo |
| `categoria_dispositivo` — `tipo_dispositivo` | 1 : N | Una categoría agrupa varios tipos |
| `tipo_dispositivo` — `dispositivo` | 1 : N | Un tipo clasifica varios dispositivos |
| `tipo_dispositivo` — `variable` | N : M | Un tipo sensa varias variables y una variable es sensada por varios tipos |
| `dispositivo` — `instalacion_dispositivo` | 1 : N | Un dispositivo acumula varias instalaciones a lo largo del tiempo |
| `campo` — `instalacion_dispositivo` | 1 : N | Un campo aloja varios dispositivos instalados |
| `sector` — `instalacion_dispositivo` | 1 : N | Un sector aloja varios dispositivos instalados |
| `pivote` — `instalacion_dispositivo` | 1 : N | Un pivote aloja varios dispositivos instalados |
| `dispositivo` — `medicion` | 1 : N | Un dispositivo genera muchas mediciones |
| `gateway` — `medicion` | 1 : N | Un gateway recibe muchas mediciones |
| `variable` — `regla_alarma` | 1 : N | Una variable es evaluada por varias reglas; cada regla evalúa una sola variable |
| `regla_alarma` — `dispositivo` | N : M | Una regla se aplica a varios dispositivos y un dispositivo tiene varias reglas |
| `regla_alarma` — `evento_alarma` | 1 : N | Una regla genera muchos eventos a lo largo del tiempo |
| `medicion` — `evento_alarma` | 1 : 0..N | Una medición puede no generar ningún evento, o generar varios |
| `perfil` — `usuario` | 1 : N | Un perfil agrupa varios usuarios; cada usuario tiene un solo perfil |

Las tres relaciones que vinculan `instalacion_dispositivo` con `campo`, `sector` y `pivote` son excluyentes entre sí: cada instalación participa de exactamente una de ellas. La notación entidad-relación no permite expresar esa exclusividad, por lo que se enuncia como restricción del dominio en la sección siguiente.

## 4.5 Restricciones del dominio

Las siguientes reglas surgen del negocio y deben cumplirse con independencia de la tecnología. Se indica además con qué mecanismo quedan garantizadas en la implementación, lo que permite distinguir las que el motor verifica por sí mismo de las que hoy dependen de la aplicación.

**Organización del establecimiento**

| Restricción | Cómo se garantiza |
| --- | --- |
| Un lote pertenece a un único campo | Clave foránea obligatoria |
| Un sector pertenece a un único lote | Clave foránea obligatoria |
| Un pivote pertenece a un único campo | Clave foránea obligatoria |
| Un pivote riega un solo lote a la vez | Índice único parcial sobre las asignaciones activas |
| Un lote es regado por un solo pivote a la vez | Sin mecanismo declarativo (ver nota) |

**Dispositivos**

| Restricción | Cómo se garantiza |
| --- | --- |
| Un dispositivo pertenece a un único tipo | Clave foránea obligatoria |
| Un tipo de dispositivo pertenece a una única categoría | Clave foránea obligatoria |
| Un tipo de dispositivo sensa al menos una variable | Sin mecanismo declarativo |
| Un dispositivo tiene como máximo una instalación activa | Índice único parcial |
| Cada instalación se asocia a exactamente un campo, sector o pivote | `CHECK` de exclusividad |
| Un dispositivo sin instalación activa no puede estar encendido | Sin mecanismo declarativo |
| Un dispositivo apagado no puede estar conectado | Sin mecanismo declarativo |
| Un dispositivo conectado debe estar encendido y tener instalación activa | Sin mecanismo declarativo |

**Mediciones**

| Restricción | Cómo se garantiza |
| --- | --- |
| Cada medición pertenece a un único dispositivo | Clave foránea obligatoria |
| Cada medición es recibida por un único gateway | Clave foránea obligatoria |
| Una medición no admite campos nulos | Parcial: `rssi` y `snr` admiten nulos |

**Alarmas**

| Restricción | Cómo se garantiza |
| --- | --- |
| Una regla de alarma define al menos un umbral | `CHECK` sobre umbral inferior y superior |
| Una regla de alarma evalúa una única variable | Clave foránea obligatoria |
| La variable de la regla debe pertenecer al tipo de dispositivo sobre el que se aplica | Sin mecanismo declarativo |

**Usuarios**

| Restricción | Cómo se garantiza |
| --- | --- |
| Existen únicamente los perfiles Operador, Configurador y Administrador | `CHECK` sobre el nombre del perfil |
| Todo usuario tiene un perfil asignado | Clave foránea obligatoria |
| Ningún usuario tiene más de un perfil | Cardinalidad del modelo |
| Un usuario no admite datos nulos | Restricciones `NOT NULL` |

**Nota sobre las restricciones sin mecanismo declarativo.** Las reglas marcadas de ese modo comparten una característica: involucran más de una fila o más de una tabla, y por lo tanto no son expresables con una clave foránea ni con un `CHECK` de fila. La coherencia entre `estado_operativo`, `estado_comunicacion` y la existencia de una instalación activa, o la pertenencia de una variable al tipo de dispositivo de la regla, requieren disparadores (`TRIGGER`) o validación en la capa de aplicación. El caso del lote regado por un solo pivote a la vez es distinto y más simple: el índice único parcial existente restringe el pivote pero no el lote, de modo que alcanza con un segundo índice análogo sobre el lote para cubrirlo. Estas restricciones se enuncian acá por pertenecer al dominio; su implementación se retoma en las secciones 8 y 13.

## 4.6 Decisiones de modelado

**Las relaciones N:M se representan como tales, sin resolver.** El diagrama muestra `tipo_dispositivo — variable` y `regla_alarma — dispositivo` como relaciones muchos a muchos, sin introducir entidades intermedias. Resolverlas mediante tablas asociativas es una decisión del modelo lógico (sección 5), no del dominio: la afirmación "un tipo de dispositivo sensa varias variables y una variable es sensada por varios tipos" es cierta independientemente de cómo se almacene. Se adoptó N:M y no 1:N porque `batería` es sensada por todos los tipos de dispositivo, y modelarla como 1:N obligaría a repetir esa variable por cada tipo, con la consiguiente redundancia y las anomalías de actualización asociadas. Esta es la razón por la que el modelo conceptual tiene 16 entidades y el esquema físico 18 tablas.

**`instalacion_dispositivo` y `asignacion_pivote` son entidades, no relaciones.** Ambas podrían parecer simples vínculos entre dos entidades, pero tienen atributos propios (`fecha_inicio`, `fecha_fin`) que no pertenecen a ninguno de los extremos: la fecha en que un dispositivo fue instalado no es un atributo del dispositivo ni del sector, sino del hecho de la instalación. Son entidades asociativas, y modelarlas así es lo que permite conservar el historial en lugar de solo el estado actual — un requisito explícito del caso (secciones 1.4 y 2.7).

**La instalación polimórfica se representa como tres relaciones excluyentes.** Un dispositivo se instala en un campo, un sector o un pivote. En el diagrama esto aparece como tres relaciones opcionales desde `instalacion_dispositivo`, porque la notación entidad-relación no puede expresar la exclusividad entre ellas. La regla "exactamente una de las tres" es una restricción del dominio (sección 4.5) y se garantiza en la implementación mediante un `CHECK` (sección 8.2). Se documenta acá para que la lectura del diagrama no induzca a pensar que una instalación puede apuntar a los tres lugares a la vez.

**Los valores medidos no se modelan como una relación entre `medicion` y `variable`.** El relevamiento de dominio enuncia que "una medición tiene múltiples variables". No se incorporó como relación explícita porque las variables que una medición puede contener quedan determinadas por el camino `medicion → dispositivo → tipo_dispositivo → variable`, que ya está en el modelo: agregar un vínculo directo duplicaría esa información y sugeriría una entidad de detalle que el diseño deliberadamente no tiene. El contenido variable de cada medición se representa como el atributo `valores_medidos`, cuya justificación como JSONB se desarrolla en las secciones 6 y 8.2.

---

# 5. Modelo de implementación según la tecnología elegida

La solución se implementa sobre una base de datos relacional —PostgreSQL, con la extensión TimescaleDB para la serie temporal de mediciones—, por lo que el modelo de implementación es un **modelo lógico relacional**. La justificación de esa elección tecnológica, comparada con las alternativas NoSQL y vectoriales, se desarrolla en la sección 7.

Esta sección presenta el modelo lógico derivado del modelo conceptual de la sección 4: tablas, columnas, claves primarias y foráneas, restricciones de integridad y resolución de las relaciones muchos a muchos. Los aspectos que dependen del motor —el particionado de la hipertabla, los índices GIN sobre JSONB y los índices únicos parciales— pertenecen al modelo físico y se tratan en la sección 8.

## 5.1 Del modelo conceptual al modelo lógico

La transformación del modelo conceptual al relacional siguió reglas sistemáticas, aplicadas de manera uniforme:

| Construcción conceptual | Traducción relacional |
| --- | --- |
| Entidad | Tabla, con identificador propio como clave primaria |
| Atributo | Columna con su tipo de dato y obligatoriedad |
| Relación 1:N | Clave foránea en el lado "muchos" |
| Relación N:M | Tabla asociativa con clave primaria compuesta (sección 5.4) |
| Entidad asociativa | Tabla propia, conservando sus atributos y su identificador |
| Relaciones excluyentes | Claves foráneas opcionales más una restricción que admite exactamente una |

El resultado son 18 tablas: las 16 entidades del modelo conceptual más las 2 tablas asociativas que resuelven las relaciones muchos a muchos.

Un punto que conviene aclarar es por qué `instalacion_dispositivo` y `asignacion_pivote` se traducen como tablas propias y no se absorben en otra. Aunque vinculan dos entidades, tienen atributos que no pertenecen a ninguno de los extremos (`fecha_inicio`, `fecha_fin`) y su cardinalidad es 1:N respecto de ambos lados: un dispositivo acumula varias instalaciones a lo largo del tiempo, y un sector aloja varios dispositivos. Absorberlas obligaría a guardar solo la ubicación vigente y perder el historial, que es un requisito del caso (secciones 1.4 y 2.7).

## 5.2 Diagrama lógico relacional

![Modelo lógico relacional](modelo_logico.png)

Cada tabla lista sus columnas con el tipo de dato, y marca las claves primarias (`PK`), foráneas (`FK`) y de unicidad (`UK`). Las columnas anotadas como `nullable` son las tres claves foráneas excluyentes de `instalacion_dispositivo`; el resto de las claves foráneas es obligatorio.

Al igual que el modelo conceptual, el diagrama se mantiene como fuente Mermaid versionada en `docs/modelo_logico.mmd` y se exporta a `docs/modelo_logico.png`.

## 5.3 Tablas y claves

El modelo comprende 18 tablas, organizadas en los mismos cinco bloques del modelo conceptual.

| Tabla | Clave primaria | Claves foráneas |
| --- | --- | --- |
| `campo` | `id_campo` | — |
| `lote` | `id_lote` | `id_campo` |
| `sector` | `id_sector` | `id_lote` |
| `pivote` | `id_pivote` | `id_campo` |
| `asignacion_pivote` | `id_asignacion` | `id_pivote`, `id_lote` |
| `categoria_dispositivo` | `id_categoria` | — |
| `tipo_dispositivo` | `id_tipo` | `id_categoria` |
| `variable` | `id_variable` | — |
| `tipo_variable` | (`id_tipo_dispositivo`, `id_variable`) | ambas columnas |
| `dispositivo` | `id_dispositivo` | `id_tipo` |
| `instalacion_dispositivo` | `id_instalacion` | `id_dispositivo`, y `id_campo` / `id_sector` / `id_pivote` (opcionales y excluyentes) |
| `gateway` | `id_gateway` | — |
| `medicion` | (`id_medicion`, `fecha_hora`) | `id_dispositivo`, `id_gateway` |
| `regla_alarma` | `id_regla` | `id_variable` |
| `alarma_dispositivo` | (`id_regla_alarma`, `id_dispositivo`) | ambas columnas |
| `evento_alarma` | `id_evento_alarma` | `id_regla` |
| `perfil` | `id_perfil` | — |
| `usuario` | `id_usuario` | `id_perfil` |

**Sobre la clave primaria de `medicion`.** Es la única tabla cuya clave primaria no responde a una decisión del modelo lógico sino a un requisito de la tecnología: al particionarse por tiempo, la columna de particionado debe formar parte de toda clave primaria o índice único de la tabla. De ahí que la clave sea (`id_medicion`, `fecha_hora`) y no `id_medicion` a secas. Es un caso de restricción del modelo físico que se refleja hacia arriba, y se detalla en la sección 8.2.

**Uso de identificadores subrogados.** Todas las tablas usan identificadores numéricos generados por el motor en lugar de claves naturales. `dispositivo` es el caso donde la alternativa era más tentadora: `numero_serie` y `dev_eui` identifican unívocamente un equipo. Se optó por el subrogado porque son datos administrados por el fabricante y por la red LoRaWAN, fuera del control del sistema; si un equipo se reaprovisiona o se corrige una carga errónea, cambiar una clave natural obligaría a propagar el cambio a todas las mediciones asociadas.

## 5.4 Resolución de las relaciones muchos a muchos

Las dos relaciones N:M del modelo conceptual se resuelven con tablas asociativas de clave primaria compuesta:

| Relación conceptual | Tabla asociativa | Clave primaria |
| --- | --- | --- |
| `tipo_dispositivo` — `variable` | `tipo_variable` | (`id_tipo_dispositivo`, `id_variable`) |
| `regla_alarma` — `dispositivo` | `alarma_dispositivo` | (`id_regla_alarma`, `id_dispositivo`) |

**Por qué clave compuesta y no un identificador propio.** Ambas tablas expresan la existencia de un vínculo y nada más: no tienen atributos adicionales ni son referenciadas por terceras tablas. Con clave primaria compuesta, el par no puede repetirse — sería imposible declarar dos veces que un mismo tipo de dispositivo sensa la misma variable. Agregar un identificador subrogado obligaría a definir además una restricción de unicidad sobre las dos columnas para conseguir la misma garantía, sumando una columna que nadie usaría.

**Qué se gana con `tipo_variable`.** Es la tabla que permite que `variable` funcione como catálogo compartido. `batería`, que todos los tipos de dispositivo sensan, existe como una única fila referenciada desde cada tipo, en lugar de repetirse tantas veces como tipos haya. Sin esta tabla, el modelo caería en la redundancia descrita en la sección 4.6.

**Qué se gana con `alarma_dispositivo`.** Permite definir una regla una sola vez y aplicarla a un conjunto de dispositivos. Una regla de "batería baja" con su umbral se escribe una vez y se asocia a todos los equipos que corresponda; cambiar el umbral es modificar una fila, no recorrer el parque de dispositivos.

## 5.5 Restricciones de integridad

El modelo lógico traslada las reglas del dominio (sección 4.5) a restricciones verificables por el motor. Se agrupan en cuatro mecanismos:

**Integridad de entidad.** Toda tabla tiene clave primaria. Las tablas asociativas la forman por composición de sus dos claves foráneas; el resto usa un identificador propio. `usuario.email` lleva además una restricción de unicidad, porque es el identificador con el que las personas inician sesión y admitir duplicados haría ambiguo el acceso.

**Integridad referencial.** Todas las claves foráneas del modelo son obligatorias salvo las tres de `instalacion_dispositivo` hacia `campo`, `sector` y `pivote`, que son opcionales por construcción: exactamente una de ellas está presente en cada fila. Esa obligatoriedad generalizada es deliberada — un lote sin campo, una medición sin dispositivo o un usuario sin perfil son estados que el dominio no admite, y dejar la columna nullable habilitaría representarlos.

**Integridad de dominio.** Se restringen por `CHECK` los atributos con conjunto de valores acotado (`dispositivo.estado_operativo`, `dispositivo.estado_comunicacion`, `perfil.nombre`), la exclusividad de la instalación polimórfica y la exigencia de que una regla de alarma defina al menos un umbral.

**Reglas no expresables declarativamente.** Como se detalla en la sección 4.5, un subconjunto de las restricciones del dominio involucra varias filas o varias tablas y no puede resolverse con claves ni con `CHECK` de fila. Requieren disparadores o validación en la capa de aplicación, y se retoman en las secciones 8 y 13.

La formulación concreta de cada restricción en el DDL, junto con los índices que las respaldan, se desarrolla en la sección 8.2.

## 5.6 Cobertura de los patrones de consulta

El modelo se validó contra las necesidades de consulta enunciadas en la sección 1.4, verificando que cada una tenga un camino de acceso resuelto por la estructura.

| Necesidad | Camino de acceso |
| --- | --- |
| Última medición de un dispositivo | `medicion` filtrada por `id_dispositivo`, ordenada por `fecha_hora` |
| Histórico de una variable en un rango de fechas | `medicion` por `id_dispositivo` y rango de `fecha_hora`, extrayendo la clave correspondiente de `valores_medidos` |
| Inventario de dispositivos por ubicación | `instalacion_dispositivo` activa (`fecha_fin IS NULL`) unida a `campo`, `sector` o `pivote` |
| Estado operativo de la red de dispositivos | `dispositivo` por `estado_operativo` / `estado_comunicacion`, unida a su instalación activa |
| Calidad del enlace y pérdida de mensajes | `medicion` agregando `rssi`, `snr` y las discontinuidades de `contador_mensajes` por dispositivo |
| Alarmas registradas en un período | `evento_alarma` por `fecha_hora`, unida a `regla_alarma` y a la `variable` evaluada |
| Reglas de alarma aplicables a un dispositivo | `alarma_dispositivo` por `id_dispositivo`, unida a `regla_alarma` |
| Qué pivote regaba un lote en una fecha dada | `asignacion_pivote` por `id_lote`, filtrando el intervalo que contiene esa fecha |

**Una consecuencia del diseño que conviene explicitar.** Consultar mediciones por ubicación —"todas las lecturas del sector B"— no se resuelve con una clave foránea directa, porque `medicion` no guarda dónde estaba el dispositivo: guarda qué dispositivo la generó. La ubicación se obtiene navegando a `instalacion_dispositivo` y seleccionando la instalación vigente en el momento de la medición, lo que constituye una unión temporal (la fecha de la medición debe caer dentro del intervalo `fecha_inicio`–`fecha_fin`).

Es el precio de conservar el historial: si un dispositivo se traslada de sector, las mediciones antiguas siguen atribuidas al sector donde efectivamente se tomaron, en lugar de reasignarse retroactivamente al nuevo. La alternativa —copiar la ubicación dentro de cada medición— haría la consulta más directa a costa de duplicar el dato y de introducir el riesgo de inconsistencia. Ese compromiso se retoma en las secciones 6 y 14.

---

# 8. Implementación mínima realizada

## 8.1 Esquema físico
El esquema físico se implementó en PostgreSQL 16 con la extensión TimescaleDB, en `db/estructura/01_create_tables.sql`. El script define 15 tablas, organizadas en cinco bloques:

1. **Organización del establecimiento**: `campo`, `lote`, `sector`, `pivote`, `asignacion_pivote`.
2. **Dispositivos IoT**: `categoria_dispositivo`, `tipo_dispositivo`, `variable`, `tipo_variable`, `dispositivo`, `instalacion_dispositivo`.
3. **Mediciones**: `gateway`, `medicion`.
4. **Alarmas**: `regla_alarma`, `evento_alarma`, `alarma_dispositivo`.
5. **Usuarios**: `perfil`, `usuario`.

El script se ejecuta sobre una base recién creada; no usa `IF NOT EXISTS` porque se apoya en el mecanismo de inicialización de la imagen de Postgres (ver 8.3), que solo corre una vez, sobre un volumen vacío.

## 8.2 Decisiones de diseño reflejadas en el DDL

**Hipertabla TimescaleDB para `medicion`.** La tabla de mediciones se convierte en hipertabla mediante `create_hypertable('medicion', 'fecha_hora')`, particionando internamente por tiempo. Es la tabla con mayor volumen esperado (una fila por dispositivo y transmisión), por lo que necesita el patrón de escritura y purga que ofrece TimescaleDB en lugar de una tabla relacional simple.

**JSONB para `valores_medidos`.** Cada tipo de dispositivo mide un conjunto distinto de variables (una estación meteorológica no mide lo mismo que una sonda de suelo). En vez de modelar una columna por variable o una tabla EAV, `medicion.valores_medidos` es JSONB, y se indexa con GIN (`ix_medicion_valores_gin`) para soportar filtros por contenido (operador `@>`) sin escanear toda la tabla.

**Instalación polimórfica de dispositivos.** Un dispositivo se instala en un campo, un sector o un pivote, nunca en más de uno a la vez. Se modeló con tres columnas FK nullable (`id_campo`, `id_sector`, `id_pivote`) en `instalacion_dispositivo` más un `CHECK` que exige que exactamente una esté presente:

```sql
CHECK (
    (id_campo IS NOT NULL)::INTEGER
    + (id_sector IS NOT NULL)::INTEGER
    + (id_pivote IS NOT NULL)::INTEGER = 1
)
```

Se prefirió esto a una tabla `ubicacion` genérica con `tipo` + `id_referencia` porque mantiene las FK reales de PostgreSQL (integridad referencial verificada por el motor), a costa de tener tres columnas nullable en vez de dos.

**Historial temporal.** `instalacion_dispositivo` y `asignacion_pivote` registran `fecha_inicio`/`fecha_fin`, donde `fecha_fin IS NULL` marca el registro activo. Un índice único parcial (`WHERE fecha_fin IS NULL`) garantiza que un mismo dispositivo o pivote no tenga más de una fila activa a la vez, sin depender de que la aplicación lo respete:

```sql
CREATE UNIQUE INDEX ux_instalacion_dispositivo_activa
    ON instalacion_dispositivo(id_dispositivo)
    WHERE fecha_fin IS NULL;
```

**Restricciones de integridad adicionales.** `regla_alarma` exige al menos un umbral definido (`umbral_inferior IS NOT NULL OR umbral_superior IS NOT NULL`); `dispositivo` restringe sus estados a valores válidos (`estado_operativo IN ('on', 'off')`, etc.) mediante `CHECK`; `usuario.email` es `UNIQUE`; las relaciones N:M (`tipo_variable`, `alarma_dispositivo`) se resuelven con tablas intermedias de clave primaria compuesta.

## 8.3 Entorno de ejecución

El entorno se levanta con `docker-compose.yml`, usando la imagen `timescale/timescaledb:latest-pg16`. El script `01_create_tables.sql` se monta en `/docker-entrypoint-initdb.d/`, que Postgres ejecuta automáticamente la primera vez que arranca sobre un volumen vacío — no hace falta correr el DDL a mano. Las credenciales y el nombre de la base se parametrizan por variables de entorno (`.env`, a partir de `.env.example`) para no versionar contraseñas.

Con esto, levantar el esquema completo desde cero se reduce a `docker compose up -d`.

---

# 9. Datos de ejemplo utilizados

## 9.1 Generación de datos sintéticos

Los datos de ejemplo se generan con un script en Python (`db/datos/generar_datos.py` + `db/datos/main.py`), usando `psycopg2` para insertar contra la base y `Faker` (locale `es_AR`) para los datos con apariencia realista (nombres, emails, ubicaciones). El script puebla las 15 tablas del esquema, respetando el orden de dependencias entre ellas y las restricciones definidas en el DDL.

## 9.2 Volumen generado

| Entidad | Cantidad |
| --- | --- |
| Campos | 3 |
| Lotes | 9 (3 por campo) |
| Sectores | 18 (2 por lote) |
| Pivotes | 6 (2 por campo) |
| Asignaciones de pivote | 6 |
| Categorías y tipos de dispositivo | según catálogo fijo (suelo, riego, meteorológico) |
| Dispositivos | 3 por tipo |
| Instalaciones de dispositivo | 1 por dispositivo (activa) |
| Gateways | 2 |
| Mediciones | 800 |
| Reglas de alarma | 4 |
| Eventos de alarma | 3 por regla (12 en total) |
| Usuarios | 6 |

Son volúmenes pequeños a propósito: alcanzan para validar entidades, relaciones y restricciones (la consigna no exige un dataset real ni de gran escala), sin complicar la verificación manual de los resultados.

## 9.3 Coherencia de los datos generados

Los datos no son aleatorios sin criterio: respetan la semántica del dominio.

- **`valores_medidos` varía según el tipo de dispositivo.** Una sonda de suelo genera JSON con humedad/temperatura/conductividad por nivel; una estación meteorológica genera temperatura, humedad relativa, viento, radiación y precipitación, coherente con lo relevado en la sección 2.2.
- **Las reglas de alarma tienen umbrales con sentido físico.** Por ejemplo, la regla de batería baja dispara con `umbral_inferior`, no `umbral_superior` (un valor de batería por debajo de un límite es el caso anómalo). Esto se verificó revisando los `valor_detectado` generados en `evento_alarma` contra el umbral de cada regla.
- **El historial temporal es consistente.** Cada dispositivo y cada pivote tiene exactamente una instalación/asignación activa (`fecha_fin IS NULL`) al finalizar la carga, reforzado por los índices únicos parciales del DDL.

## 9.4 Exportación de ejemplo

El script exporta una muestra de 100 mediciones (join contra `dispositivo` y `tipo_dispositivo`) a `data/ejemplos/mediciones.csv`, serializando `valores_medidos` como JSON válido (`json.dumps`) en lugar de la representación de string por defecto de Python, para que el CSV sea reutilizable por otras herramientas. 

# 10 Contexto de datos de prueba

Las consultas de esta sección se evaluaron sobre una carga sintética generada por el procedimiento `popular_tablas()`, con mediciones distribuidas en una ventana temporal de 1 año.  
La carga incluye múltiples dispositivos y gateways activos, con variabilidad por tipo de sensor y valores JSONB acordes al dominio (`humedad_suelo`, `temperatura_suelo`, `caudal`, `bateria`, etc.).  
Este enfoque permite validar consultas operativas y analíticas en un escenario más cercano a producción.

## 10.1 Humedad promedio por lote en los últimos 7 días

**Consulta:**
```sql
SELECT
    l.id_lote,
    l.nombre AS lote,
    ROUND(AVG((m.valores_medidos ->> 'humedad_suelo')::numeric), 2) AS humedad_promedio
FROM medicion m
JOIN dispositivo d
    ON d.id_dispositivo = m.id_dispositivo
JOIN instalacion_dispositivo i
    ON i.id_dispositivo = d.id_dispositivo
JOIN sector s
    ON s.id_sector = i.id_sector
JOIN lote l
    ON l.id_lote = s.id_lote
WHERE i.fecha_fin IS NULL
  AND m.fecha_hora >= NOW() - INTERVAL '7 days'
GROUP BY l.id_lote, l.nombre
ORDER BY humedad_promedio ASC;
```

Qué pregunta responde:
¿En qué lotes el nivel de humedad del suelo está más bajo en la última semana?

Por qué es útil:
Permite detectar zonas con riesgo de sequía o estrés hídrico, y facilita la toma de decisiones sobre riego y asignación de recursos. Es una consulta clave para la gestión operativa del campo.

## 10.2 Pivotes activos y su lote asignado

**Consulta:**
```sql
SELECT
    p.id_pivote,
    p.nombre AS pivote,
    l.id_lote,
    l.nombre AS lote,
    ap.fecha_inicio,
    ap.fecha_fin
FROM asignacion_pivote ap
JOIN pivote p
    ON p.id_pivote = ap.id_pivote
JOIN lote l
    ON l.id_lote = ap.id_lote
WHERE ap.fecha_fin IS NULL
ORDER BY p.id_pivote;
```

Qué pregunta responde:
¿Qué pivote está regando cada lote en este momento?

Por qué es útil:
Es relevante para coordinar la operación de riego y para verificar que la infraestructura activa esté correctamente asociada al lote correspondiente. También ayuda a detectar errores de configuración o cambios de asignación.

## 10.3 Dispositivos con batería baja

**Consulta:**
```sql
WITH ultimas_mediciones AS (
    SELECT
        m.id_dispositivo,
        m.valores_medidos,
        ROW_NUMBER() OVER (
            PARTITION BY m.id_dispositivo
            ORDER BY m.fecha_hora DESC
        ) AS rn
    FROM medicion m
)
SELECT
    d.id_dispositivo,
    td.nombre AS tipo_dispositivo,
    (um.valores_medidos ->> 'bateria')::numeric AS bateria_actual
FROM ultimas_mediciones um
JOIN dispositivo d
    ON d.id_dispositivo = um.id_dispositivo
JOIN tipo_dispositivo td
    ON td.id_tipo = d.id_tipo
WHERE um.rn = 1
  AND (um.valores_medidos ->> 'bateria')::numeric < 20
ORDER BY bateria_actual ASC;
```

Qué pregunta responde:
¿Qué dispositivos tienen batería crítica y requieren mantenimiento?

Por qué es útil:
Los sensores deben mantenerse operativos para asegurar continuidad en la medición. Esta consulta permite anticipar fallas y reducir el riesgo de pérdida de datos o interrupciones del monitoreo.

## 10.4 Eventos de alarma por regla

**Consulta:**
```sql
SELECT
    ra.id_regla,
    ra.descripcion,
    COUNT(*) AS total_eventos
FROM evento_alarma ea
JOIN regla_alarma ra
    ON ra.id_regla = ea.id_regla
WHERE ea.fecha_hora >= NOW() - INTERVAL '30 days'
GROUP BY ra.id_regla, ra.descripcion
ORDER BY total_eventos DESC;
```

Qué pregunta responde:
¿Cuáles son las alarmas más frecuentes en los últimos 30 días?

Por qué es útil:
Permite medir la criticidad del sistema y priorizar intervenciones. Es una consulta orientada a la gestión operativa, porque ayuda a identificar patrones de anomalías repetidas y posibles problemas estructurales.

## 10.5 Promedio de temperatura y humedad por sector

**Consulta:**
```sql
SELECT
    s.id_sector,
    s.nombre AS sector,
    l.id_lote,
    l.nombre AS lote,
    ROUND(AVG((m.valores_medidos ->> 'temperatura_suelo')::numeric), 2) AS temp_promedio,
    ROUND(AVG((m.valores_medidos ->> 'humedad_suelo')::numeric), 2) AS humedad_promedio
FROM medicion m
JOIN dispositivo d
    ON d.id_dispositivo = m.id_dispositivo
JOIN instalacion_dispositivo i
    ON i.id_dispositivo = d.id_dispositivo
JOIN sector s
    ON s.id_sector = i.id_sector
JOIN lote l
    ON l.id_lote = s.id_lote
WHERE i.fecha_fin IS NULL
  AND m.fecha_hora >= NOW() - INTERVAL '24 hours'
GROUP BY s.id_sector, s.nombre, l.id_lote, l.nombre
ORDER BY l.nombre, s.nombre;
```

Qué pregunta responde:
¿En qué sectores se registran condiciones más extremas de temperatura o humedad?

Por qué es útil:
Es clave para identificar zonas con comportamiento anómalo dentro del lote, optimizar el riego y detectar condiciones que puedan afectar el rendimiento del cultivo.

## 10.6 Valor del conjunto de consultas

El conjunto de consultas cubre los patrones operativos y analíticos más relevantes del caso de uso:

- **Agregación temporal y por entidad agrícola** (`GROUP BY` por lote/sector y filtros por ventana de tiempo de 24 h, 7 días y 30 días).
- **Trazabilidad de infraestructura activa** (estado vigente de pivotes e instalaciones con `fecha_fin IS NULL`).
- **Detección de condiciones críticas** (batería baja y frecuencia de alarmas).
- **Uso de datos semiestructurados** (`JSONB`) con extracción de variables desde `valores_medidos`.
- **Funciones de ventana** (`ROW_NUMBER`) para obtener la última medición por dispositivo.

En conjunto, estas consultas validan que la solución no solo almacena mediciones, sino que también soporta monitoreo operativo, mantenimiento preventivo y análisis para toma de decisiones.

## 10.7 Validación de rendimiento

Para validar desempeño se ejecutaron pruebas sobre la carga sintética generada por `popular_tablas()`, con mediciones distribuidas en una ventana de 1 año y múltiples dispositivos/gateways.

### 10.7.1 Metodología

1. Se ejecutó EXPLAIN ANALYZE sobre consultas representativas.
2. Se comparó el tiempo antes y después de crear índices adicionales de optimización en columnas de claves foráneas y filtros temporales.
3. Se evaluaron especialmente consultas con:
   - joins sobre tablas de alta cardinalidad,
   - filtros por fecha,
   - agregaciones.

### 10.7.2 Resultado resumido

| Consulta | Tiempo sin índices adicionales | Tiempo con índices | Mejora |
| --- | --- | --- | --- |
| 10.1 Humedad promedio por lote (7 días) | 22.798 ms | 2.360 ms | 89.65 % |
| 10.3 Dispositivos con batería baja | 44.520 ms | 38.566 ms | 13.37 % |
| 10.5 Temperatura/humedad por sector (24 h) | 5.147 ms | 0.999 ms | 80.59 % |

### 10.7.3 Conclusión de rendimiento

La estrategia de índices adicionales reduce el costo de joins y filtros temporales, y mejora la latencia de consultas operativas frecuentes.  
La mejora no es homogénea entre consultas: es muy alta en 10.1 y 10.5, y moderada en 10.3, donde el costo dominante está asociado al ordenamiento y al procesamiento de la ventana para obtener la última medición por dispositivo.  
En este contexto, el modelo resulta apto para monitoreo continuo y análisis sobre volúmenes altos de mediciones.

---

# 11. Datos semiestructurados, no estructurados y vectoriales

La consigna pide analizar los tres tipos de datos que exceden lo puramente relacional y decidir, para
cada uno, si el caso justifica incorporarlos y con qué tecnología. La respuesta es distinta en cada
caso: los semiestructurados son centrales y ya están implementados; los no estructurados son
previsibles pero quedan fuera del alcance, con una propuesta de cómo incorporarlos; los vectoriales
**no se justifican**, y esta sección explica por qué.

## 11.1 Datos semiestructurados

### Qué son en este sistema

Las mediciones. Cada tipo de dispositivo sensa un conjunto distinto de variables, y esa
heterogeneidad es intrínseca al dominio, no un defecto del relevamiento:

| Tipo de dispositivo | Variables por medición |
| --- | --- |
| Sonda de suelo | humedad, temperatura y conductividad eléctrica **por cada uno de 6 niveles** → 18 valores |
| Estación meteorológica | temperatura, humedad relativa, velocidad y dirección del viento, radiación solar, precipitación → 6 valores |
| Caudalímetro de riego | caudal, presión → 2 valores |

Un mismo `INSERT` de la ingesta tiene que poder registrar 2 o 18 valores según qué dispositivo
transmitió, y mañana un tipo de sensor nuevo traerá otro conjunto.

### Cómo se resuelve

Con una columna `JSONB` (`medicion.valores_medidos`) y un índice GIN que permite filtrar por contenido
con el operador `@>`. Las decisiones de modelado que sostienen esa elección —y la comparación contra
las alternativas relacionales puras— corresponden a las secciones §5 y §6 (`A-07`, `A-08`). Desde la
perspectiva de esta sección interesan tres consecuencias:

**Es la decisión que hace posible la escala del catálogo.** Las alternativas relacionales tienen costos
concretos: una tabla ancha con una columna por variable obliga a un `ALTER TABLE` sobre la tabla más
grande del sistema cada vez que aparece un tipo de sensor, y deja la mayoría de las columnas en `NULL`;
un modelo entidad-atributo-valor (una fila por variable medida) multiplica por 18 el número de filas de
la tabla que ya es la más voluminosa.

**El precio es la validación.** `JSONB` no valida claves ni tipos: nada impide guardar
`{"humedda": "veinte"}`. La integridad que en una columna daría un `CHECK` acá tiene que garantizarla
la capa de ingesta, o agregarse como restricción explícita sobre el contenido del documento. Es un
compromiso asumido, no un descuido, y conviene que el informe lo diga.

**Habilita el catálogo de variables.** Las tablas `variable` y `tipo_variable` documentan qué debería
contener el `JSONB` de cada tipo de dispositivo. El esquema queda declarado en el modelo relacional
aunque el dato se guarde sin esquema fijo, que es lo que mantiene el sistema legible.

## 11.2 Datos no estructurados

El relevamiento (§2.3) los declara «no contemplados por ahora». Es correcto para el alcance de este
trabajo, pero conviene decir **cuáles serían** y cómo se incorporarían, porque es previsible que
aparezcan en cuanto la plataforma crezca:

| Dato | Origen | Uso previsible |
| --- | --- | --- |
| Imágenes satelitales e índices de vegetación (NDVI) | Sentinel-2, Landsat | Correlacionar vigor del cultivo con humedad de suelo |
| Fotografías aéreas de dron | Vuelos periódicos sobre el lote | Detección de zonas con estrés hídrico |
| Fotos de plagas y enfermedades | Cámara del personal de campo | Diagnóstico asistido |
| Manuales y protocolos | PDF de fabricantes, protocolos agronómicos | Consulta operativa |
| Notas de campo | Texto libre del operador | Contexto de eventos no sensados |

**Cómo se incorporarían.** No dentro de PostgreSQL. El criterio es guardar el archivo en un
almacenamiento de objetos (S3, MinIO) y en la base sólo los **metadatos y la URI**: fecha, campo o lote
asociado, tipo de archivo, autor y tamaño. Las razones son de arquitectura, no de gusto:

- Un `BYTEA` con imágenes infla la base, los backups y la replicación con datos que nunca se consultan
  por contenido desde SQL.
- El acceso a un archivo es por identificador, no por predicado: no hay nada que un motor relacional
  aporte a esa operación.
- El almacenamiento de objetos es más barato y escala por separado.

La contrapartida —que es lo que hay que vigilar— es que **el archivo queda fuera del perímetro de
permisos de la base**: los `GRANT` y la RLS de la §13 no alcanzan a un objeto en S3. El control de
acceso pasa a depender de URLs firmadas con vencimiento y de la política del bucket, y esa es una
segunda superficie que hay que mantener sincronizada con la primera.

## 11.3 Datos vectoriales: por qué **no** se incorpora una base vectorial

Esta es la decisión que la consigna pide justificar explícitamente. La respuesta corta es que el caso
de uso **no tiene un problema de búsqueda por similitud** que valga la pena resolver hoy. La larga son
cinco argumentos.

### 1. La carga de trabajo real es de filtros exactos, no de similitud

Una base vectorial resuelve una pregunta muy específica: «dado un vector, encontrar los *k* más
parecidos». Las consultas que este sistema necesita responder son de otra naturaleza:

- humedad promedio del sector B en las últimas 48 horas → **rango temporal + agregación**
- dispositivos sin transmitir desde hace más de 2 horas → **comparación de fechas**
- mediciones que superaron el umbral de una regla → **filtro numérico**
- efecto del último riego sobre la humedad del lote → **`JOIN` temporal entre series y asignaciones**

Ninguna es una búsqueda por proximidad en un espacio de *embeddings*. Todas se resuelven con índices
B-tree, BRIN y GIN sobre la hipertabla, que es exactamente para lo que TimescaleDB está construido.

### 2. Los modelos previstos no usan embeddings

Los usos de IA que el trabajo contempla —predicción de necesidad de riego, detección de anomalías en
las series, estimación de evapotranspiración— son problemas clásicos de series temporales. Se
resuelven con variables derivadas (medias móviles, acumulados de lluvia, grados-día) alimentando
modelos estadísticos o de *gradient boosting*. Esas variables son **números en columnas**, no vectores
densos que haya que comparar por distancia coseno.

### 3. Los casos donde un vector serviría son periféricos y de bajo volumen

Existen, y hay que nombrarlos para que la justificación sea honesta:

- **Búsqueda semántica sobre manuales de dispositivos y protocolos agronómicos** (RAG). Volumen:
  decenas o cientos de documentos.
- **Búsqueda de situaciones análogas**: «encontrar días con un perfil de humedad y clima parecido al de
  hoy», representando una ventana de la serie como vector.

El primero es un problema documental, no del núcleo de datos. El segundo es interesante pero
exploratorio, y con el volumen actual se resuelve con una consulta de distancia euclídea sobre pocas
decenas de variables — sin índice aproximado ni motor especializado.

### 4. Un motor vectorial separado agrega costo desproporcionado

Incorporar Pinecone, Weaviate o Qdrant significa: un componente más que desplegar y monitorear, una
segunda copia de los datos, un proceso de sincronización que puede quedar desfasado, y una fuente más
de verdad a reconciliar. Todo eso para atender un uso que hoy es hipotético. La regla que se aplica en
todo el trabajo —no sumar tecnología antes de tener el problema que la justifique— se aplica también
acá.

### 5. Y el argumento decisivo: un almacén separado rompe el modelo de seguridad

Este es el punto que inclina la decisión y que conecta con la §13.

El modelo de permisos define privilegios por columna, vistas de enmascaramiento y aislamiento por
establecimiento mediante RLS. **Nada de eso viaja con los datos.** Un índice vectorial en un motor
externo es, por construcción, una copia fuera del perímetro: los *embeddings* calculados sobre
mediciones de todos los campos quedarían en un sistema que no conoce `seguridad.acceso_campo` y no
puede filtrar por usuario. Una consulta de similitud devolvería vecinos de cualquier establecimiento,
y reconstruir el aislamiento del lado del cliente es exactamente el tipo de control que la §13 descarta
por frágil.

El riesgo es el mismo que se describe en §13.8 para las aplicaciones de IA, agravado: no es que un
modelo pueda revelar un dato al que tiene acceso, es que se crea deliberadamente una réplica de los
datos sin ninguna de sus defensas.

### Qué se hace en cambio

**Si en algún momento hace falta búsqueda vectorial, se usa `pgvector` dentro de la misma instancia de
PostgreSQL.** La extensión aporta el tipo `vector` y los índices HNSW e IVFFlat, que es todo lo que los
dos casos del punto 3 necesitan a este volumen. Las ventajas son directas:

- **No agrega infraestructura.** Es una extensión, no un servicio.
- **Los embeddings quedan dentro del perímetro de seguridad**: una tabla con columna `vector` está
  sujeta a los mismos `GRANT` y a las mismas políticas RLS que cualquier otra.
- **Se pueden combinar** filtros relacionales y similitud en una sola consulta —«los 5 días más
  parecidos a hoy *dentro de los campos que este usuario puede ver*»—, que es justamente lo que un
  motor separado no puede hacer.
- Evita el problema de sincronización: el vector se calcula y se guarda en la misma transacción que el
  dato del que deriva.

El directorio `vectorial/` del repositorio contiene esta misma justificación, para que quien lo
encuentre vacío entienda que la ausencia de modelo es una decisión y no una tarea pendiente.

## 11.4 Resumen

| Tipo de dato | ¿Presente en el caso? | Decisión | Tecnología |
| --- | --- | --- | --- |
| Semiestructurado | Sí, es central | Implementado | `JSONB` + índice GIN en PostgreSQL |
| No estructurado | Previsible, fuera de alcance | Propuesta documentada | Almacenamiento de objetos + metadatos en PostgreSQL |
| Vectorial | Marginal e hipotético | **No se incorpora** | Si hiciera falta: `pgvector` en la misma instancia, nunca un motor separado |

---

# 12. Propuesta de arquitectura de datos

Esta sección describe la arquitectura de datos de punta a punta: por dónde entra cada medición,
dónde se almacena, cómo se prepara para el análisis y quién la consume. El diagrama que la
acompaña está versionado como fuente Mermaid en `docs/arquitectura.mmd` y exportado a
`docs/arquitectura.png`, siguiendo el mismo criterio que los modelos conceptual y lógico:
versionar el fuente además de la imagen permite revisar los cambios en el control de versiones.

## 12.1 Visión general

La arquitectura se organiza en cinco capas, siguiendo el recorrido del dato desde el sensor hasta el
consumidor final.

![Arquitectura de datos](arquitectura.png)

| Capa | Función | Identidad de acceso |
| --- | --- | --- |
| 1 · Origen | Dispositivos IoT y gateways LoRaWAN | — (red LoRaWAN) |
| 2 · Ingesta | Network server, cola y normalización | `rol_ingesta` |
| 3 · Operacional | PostgreSQL 16 + TimescaleDB | `rol_operador`, `rol_configurador`, `rol_administrador` |
| 4 · Analítica | Agregados continuos, retención, vista analítica | `rol_analitico` |
| 5 · Consumo | Aplicación, administración, BI y modelos de IA | según el consumidor |

Cada capa se corresponde con un rol de base de datos concreto, definido en
`db/estructura/04_roles_permisos.sql`. Esto no es casual: la arquitectura y el modelo de permisos se
diseñaron juntos, de modo que el límite entre dos capas sea también un límite de privilegios y no sólo
una línea en un diagrama.

## 12.2 Capa 1 — Origen

Los dispositivos (sondas de suelo, estaciones meteorológicas y sensores de riego) transmiten por
LoRaWAN a intervalos configurables —`dispositivo.intervalo_transmision`, típicamente 15 a 60 minutos—.
Los gateways reciben esas transmisiones y agregan los metadatos de calidad de enlace que el modelo
conserva en cada medición: `rssi`, `snr` y `contador_mensajes`.

Esos tres campos no son decorativos. `contador_mensajes` es una numeración ascendente por dispositivo,
y por lo tanto **permite detectar mensajes perdidos**: un salto en la secuencia indica que un uplink no
llegó. El relevamiento (§2.7) ya lo identifica como dato de auditoría; la arquitectura lo aprovecha
como señal de salud de la red, no sólo como dato histórico.

Característica del enlace que condiciona todo lo que sigue: LoRaWAN es de bajo ancho de banda, sin
garantía de entrega y **con posibilidad de duplicados**, porque un mismo uplink puede ser recibido por
más de un gateway.

## 12.3 Capa 2 — Ingesta

Tres componentes en cadena:

**Network server LoRaWAN** (ChirpStack o The Things Network). Desencripta los payloads y gestiona la
sesión de cada dispositivo. **Es el custodio real de las claves**: la copia que la base guarda en
`dispositivo.app_key` es inventario para provisionamiento y rotación, no el material que se usa en la
comunicación. Por eso `rol_ingesta` no tiene acceso de lectura a esa columna (§13.4).

**Broker MQTT y cola.** Desacopla la recepción de la escritura en la base. Es la decisión de diseño
más importante de esta capa y conviene justificar por qué no se escribe directo:

- Los uplinks llegan en ráfagas —los dispositivos de un mismo lote suelen transmitir en ventanas
  cercanas— y la cola amortigua el pico en lugar de trasladarlo a la base.
- Si la base se cae o se reinicia por mantenimiento, los mensajes se acumulan en la cola y se procesan
  después, en vez de perderse. Recuperar una medición perdida es imposible: el dispositivo no
  retransmite.
- Permite escribir en lotes (`batch`) en lugar de una transacción por medición, que es la diferencia
  entre miles de *commits* por minuto y unas pocas decenas.

**Servicio de normalización.** Resuelve `dev_eui` → `id_dispositivo`, arma el `JSONB` de
`valores_medidos` según el tipo de dispositivo y aplica las reglas de alarma vigentes para generar
`evento_alarma` cuando corresponde.

**Cómo se puebla hoy la base.** El procedimiento `popular_tablas()`
(`db/estructura/02_populate_tables.sql`, tarea `C-01`) genera una carga completa de prueba —50.000
mediciones distribuidas sobre un año— sin depender del script de Python. Es el sustituto práctico de
la capa de ingesta mientras no exista el conector real, y basta para validar el modelo, las consultas
y los permisos.

**Idempotencia — pendiente de coordinar con B.** Como los duplicados son parte normal del protocolo,
la ingesta debe ser idempotente. El par `(id_dispositivo, contador_mensajes)` identifica unívocamente
un uplink, así que la forma natural de garantizarlo es una restricción única sobre esas dos columnas
más `INSERT … ON CONFLICT DO NOTHING`. **El DDL actual no la tiene**, y agregarla es una modificación
sobre `01_create_tables.sql`, territorio de B. Queda anotada como recomendación, no aplicada de forma
unilateral.

## 12.4 Capa 3 — Operacional

PostgreSQL 16 con TimescaleDB. La justificación tecnológica corresponde a la §7 (`E-03`); acá sólo
interesa cómo se organiza internamente.

Cuatro conjuntos de datos con perfiles de uso muy distintos conviven en la misma instancia:

| Conjunto | Volumen | Patrón | Tablas |
| --- | --- | --- | --- |
| Series temporales | Altísimo, creciente | Escritura constante, lectura por rango de tiempo | `medicion` (hipertabla) |
| Configuración | Bajo, estable | Lectura frecuente, escritura esporádica | `campo`, `lote`, `sector`, `pivote`, `dispositivo`, `instalacion_dispositivo` |
| Alarmas | Medio | Escritura por evento, lectura por consulta | `regla_alarma`, `evento_alarma` |
| Identidad y auditoría | Bajo / creciente | Acceso restringido | `usuario`, `perfil`, `seguridad.acceso_campo`, `auditoria.registro_cambios` |

**Por qué una sola instancia y no una base por conjunto.** Las consultas que dan valor al sistema
cruzan las cuatro categorías: «humedad promedio del sector B durante el último riego» necesita
mediciones, instalación, sector, lote y asignación de pivote en una misma consulta. Separarlas en
motores distintos convertiría cada `JOIN` en una unión hecha a mano en la aplicación, y perdería las
garantías de integridad referencial que sostienen el modelo. La heterogeneidad se resuelve **dentro**
del motor: hipertabla para lo temporal, tablas relacionales para lo estable, `JSONB` para lo variable.

**Índices.** `db/indices_vistas/01_indices_fk.sql` (`C-02`) cubre las claves foráneas y los filtros
temporales más frecuentes. Es una diferencia que conviene explicitar porque suele sorprender:
PostgreSQL indexa automáticamente las claves primarias, pero **no** las foráneas, y en un modelo con
la cantidad de relaciones de éste esa ausencia se nota en cada `JOIN`.

**Auditoría transversal.** Las tablas de configuración e identidad escriben en `auditoria` mediante
triggers (§13.6). En el diagrama se representa con línea punteada porque no forma parte del flujo del
dato de negocio: es un flujo de control que corre en paralelo.

## 12.5 Capa 4 — Analítica

No es una base separada, sino una **capa lógica** dentro de la misma instancia, construida con tres
mecanismos:

- **Vistas de resumen y agregados continuos**: `v_mediciones_resumen` (`db/indices_vistas/`, tarea
  `C-02`) ya promedia humedad, temperatura y caudal por dispositivo y por hora. El paso siguiente es
  convertirla en agregado continuo de TimescaleDB, materializado y actualizado incrementalmente, para
  que un tablero de la última semana consulte el agregado en lugar de las decenas de miles de filas de
  detalle.

  Nota de seguridad, desarrollada en §13: esa vista agrega mediciones de **todos** los campos y usa el
  modo de ejecución por defecto, con lo cual no respeta la RLS. Por eso `04_roles_permisos.sql` se la
  otorga sólo a `rol_analitico`. Para que la consuma también la aplicación de monitoreo hay que
  declararla `WITH (security_invoker = true)`.
- **Compresión y retención**: detalle reciente sin comprimir para consulta operativa, histórico
  comprimido para análisis. El detalle es el terreno de la §14 (`C-04`); acá se registra que la capa
  analítica depende de esas políticas.
- **`v_medicion_analitica`**: la vista que expone mediciones con su ubicación **histórica** —la que
  tenía el dispositivo en el momento de medir, no la actual—, sin datos personales ni credenciales.

**Cuándo separar físicamente.** Mientras la carga analítica no compita con la ingesta, mantener todo en
una instancia es la opción simple y correcta. El punto de quiebre es observable: cuando las consultas
analíticas empiecen a degradar la latencia de escritura, el paso siguiente es una **réplica de sólo
lectura** que atienda a `rol_analitico` y a BI, sin cambiar el modelo ni las aplicaciones — sólo la
cadena de conexión. Se documenta el camino y no se implementa: adelantar una réplica para el volumen
actual sería complejidad sin beneficio.

## 12.6 Capa 5 — Consumidores

- **Aplicación de monitoreo** — `rol_operador` y `rol_configurador`, con RLS filtrando por campo.
- **Administración** — `rol_administrador`, único acceso a identidad y auditoría.
- **Tableros y BI** — `rol_analitico`, sobre la vista analítica.
- **Modelos de IA** — mismo rol que BI, misma superficie de acceso. Las predicciones vuelven a la
  aplicación como recomendación (línea punteada en el diagrama): el modelo sugiere, no acciona sobre el
  riego. Es una decisión de arquitectura, no una limitación técnica — un sistema que riega solo con
  base en una predicción necesita garantías que exceden el alcance de este trabajo.

## 12.7 Decisiones de arquitectura y sus compromisos

| Decisión | A favor | En contra | Por qué se toma |
| --- | --- | --- | --- |
| Un solo motor para operacional y analítico | Simplicidad, `JOIN`s directos, integridad referencial | Las cargas compiten por recursos | El volumen del caso lo admite; el camino a réplica está documentado |
| Cola entre network server y base | Absorbe picos y cortes, permite escritura por lotes | Un componente más que operar | Las mediciones perdidas no se recuperan |
| `JSONB` para `valores_medidos` | Un tipo nuevo de dispositivo no cambia el esquema | Sin validación por columna | Ver §5 y §6 (`A-07`, `A-08`) |
| Capa analítica lógica, no física | Sin sincronización ni duplicación de datos | Aislamiento de cargas limitado | Evita el problema de consistencia de un pipeline ETL |
| Ubicación histórica en la vista analítica | Los datos de entrenamiento no se corrompen al mover un sensor | `JOIN` temporal más costoso | La corrección del dato histórico no es negociable |

## 12.8 Relación con otras secciones

- **§7** justifica la elección de PostgreSQL + TimescaleDB frente a las alternativas (`E-03`).
- **§11** explica qué datos no estructurados quedan fuera de esta arquitectura y por qué no se suma un
  almacén vectorial (`D-03`).
- **§13** desarrolla el modelo de permisos que da identidad a cada capa (`D-05`).
- **§14** trata particionado, retención y rendimiento con el detalle que esta sección sólo enuncia
  (`C-04`).

---

# 13. Estrategia de seguridad, permisos y aislamiento

La implementación de todo lo que describe esta sección está en `db/estructura/04_roles_permisos.sql`,
`05_auditoria.sql` y `06_rls_aislamiento.sql`, y su comprobación en
`anexos/verificacion_seguridad.sql`, que ejercita cada control desde el rol que corresponde
(§13.9).

## 13.1 Qué hay que proteger

Antes de enumerar permisos conviene decir qué se está cuidando, porque no todos los datos del
sistema tienen el mismo valor para un atacante ni el mismo costo si se pierden.

| Activo | Dónde vive | Por qué importa |
| --- | --- | --- |
| Credenciales LoRaWAN | `dispositivo.app_key`, `at_pin`, `ota_pin` | Quien las obtiene puede **suplantar al dispositivo** y transmitir mediciones falsas |
| Datos personales | `usuario.nombre`, `apellido`, `email`, `contrasena` | Datos de personas identificables; obligaciones legales y reputacionales |
| Historial de mediciones | `medicion` | Revela superficie sembrada, régimen de riego y rendimiento de un establecimiento: información comercialmente sensible |
| Configuración de riego y alarmas | `regla_alarma`, `asignacion_pivote`, `instalacion_dispositivo` | Alterarla produce daño físico: riego indebido o alarmas silenciadas |
| Registro de auditoría | `auditoria.registro_cambios` | Es la prueba de todo lo anterior; si se puede editar, no prueba nada |

La particularidad del caso es que **la integridad pesa más que la confidencialidad**. Que se filtre
una medición de humedad es un problema; que alguien inyecte mediciones falsas de humedad es un
problema mayor, porque el sistema existe para decidir cuándo regar y una lectura falsa se traduce
en una decisión de riego equivocada sobre un cultivo real. Esto ordena las prioridades: primero la
autenticidad del dato que entra, después el control de quién lo lee.

## 13.2 Punto de partida: qué encontramos en la implementación

El modelo físico entregado en `B-01` y el generador de datos de `B-02` funcionan correctamente,
pero —como es esperable en un DDL que todavía no había pasado por la mirada de seguridad— dejaban
varios huecos. Se documentan porque la corrección de cada uno explica una decisión de esta sección.

| # | Hallazgo | Dónde | Corrección |
| --- | --- | --- | --- |
| 1 | Contraseñas almacenadas en texto plano | `01_create_tables.sql` (`usuario.contrasena TEXT`) y `generar_datos.py` (`fake.password()` insertado tal cual) | Trigger de hash bcrypt (§13.5) |
| 2 | Credenciales LoRaWAN legibles por cualquiera con `SELECT` sobre `dispositivo` | `01_create_tables.sql`, columnas `app_key`, `at_pin`, `ota_pin` | Privilegios por columna (§13.4) |
| 3 | Toda la operación corre como superusuario | `.env.example` (`DB_USER=postgres`) | Roles de mínimo privilegio (§13.3) |
| 4 | Sin registro de cambios, pese a estar comprometido en §2.7 | — | Esquema `auditoria` (§13.6) |
| 5 | Sin aislamiento entre establecimientos | — | RLS por campo (§13.7) |
| 6 | `TRUNCATE` no deja rastro de auditoría | `02_populate_tables.sql` (`popular_tablas()`) | Trigger de sentencia (§13.6) |
| 7 | pgAdmin expuesto con credenciales por defecto | `docker-compose.yml` | Ver más abajo |

El hallazgo 4 merece una aclaración: el relevamiento de datos de la §2.7 declara entre los datos de
auditoría los «cambios realizados sobre configuraciones y alarmas». Era una promesa del relevamiento
que el modelo físico no cumplía. Cerrar esa distancia no es agregar una funcionalidad extra, es
terminar de implementar lo que el propio informe dice que el sistema guarda.

Los hallazgos 6 y 7 surgieron al integrar el trabajo de C. El 6 está explicado en §13.6.

Sobre el **hallazgo 7**: `docker-compose.yml` incorporó un servicio `pgadmin` publicado en el puerto
8080 con las credenciales `admin@admin.com` / `admin` escritas en el archivo. Desde ahí se puede abrir
una conexión a la base como `postgres`, que es superusuario y por lo tanto **saltea todo lo que
describe esta sección**: los privilegios por columna, las vistas de enmascaramiento y la RLS no aplican
al dueño de las tablas. Es una herramienta de desarrollo cómoda y no hay razón para quitarla del
entorno local, pero conviene dejar constancia de tres cosas: que no debe publicarse fuera de la máquina
de desarrollo, que sus credenciales por defecto tienen que cambiar antes de cualquier despliegue
compartido, y que la conexión que se configure en pgAdmin debería usar uno de los roles de aplicación
(§13.3) y no `postgres`, para que la herramienta quede sujeta al mismo modelo de permisos que todo lo
demás.

## 13.3 Roles: identidades de persona e identidades de servicio

El dominio define tres perfiles (§1.3): Operador, Configurador y Administrador. Se implementan como
tres roles de grupo de PostgreSQL con herencia acumulativa, que reproduce la relación descripta en el
relevamiento —el Configurador hace todo lo del Operador y además configura; el Administrador, todo lo
anterior más usuarios y permisos—:

```sql
GRANT rol_operador     TO rol_configurador;
GRANT rol_configurador TO rol_administrador;
```

A esos tres se agregan **dos roles que el modelo de negocio no nombra pero la arquitectura necesita**,
porque no toda conexión a la base la hace una persona:

- **`rol_ingesta`** — el conector que recibe los uplinks del network server LoRaWAN. Sólo inserta
  mediciones y eventos de alarma. **No puede leer el histórico.** Es una decisión deliberada: si el
  conector se ve comprometido —y es el componente más expuesto, porque escucha desde internet— el
  atacante puede ensuciar datos nuevos, pero no exfiltrar años de historia.
- **`rol_analitico`** — los consumidores analíticos: tableros, notebooks y aplicaciones de IA. No
  tiene acceso a *ninguna* tabla; sólo a una vista construida para él (§13.8).

La distinción entre perfil de persona e identidad de servicio es lo que permite aplicar mínimo
privilegio de verdad. Si la ingesta se conectara con el rol del Administrador —que es lo que ocurre
hoy, donde todo usa `postgres`— cualquier compromiso del conector sería un compromiso total.

Los privilegios se otorgan **siempre a roles de grupo, nunca a personas**. Un alta o una baja se
resuelve con un `GRANT` o un `REVOKE` de pertenencia, sin volver a tocar la matriz de permisos.

### Cierre de los privilegios que PostgreSQL otorga por defecto

PostgreSQL concede al pseudo-rol `PUBLIC` —es decir, a todo rol existente y futuro— la capacidad de
conectarse a cualquier base y de usar el esquema `public`. El script empieza por revocarlo:

```sql
REVOKE ALL ON DATABASE <base> FROM PUBLIC;
REVOKE ALL ON SCHEMA public   FROM PUBLIC;
```

La diferencia entre partir de «todo permitido salvo excepciones» y «todo denegado salvo lo otorgado»
es que en el primer esquema un rol nuevo hereda acceso **por olvido**, y nadie audita un permiso que
nunca se escribió.

## 13.4 Permisos por columna: credenciales de sólo escritura

El corte de privilegios sobre `dispositivo` no es por tabla sino por columna, y es la decisión de
diseño más específica de esta sección.

| Rol | Ficha del dispositivo | `app_key`, `at_pin`, `ota_pin` |
| --- | --- | --- |
| Operador | lectura | — |
| Configurador | lectura y escritura | **escritura sin lectura** |
| Administrador | lectura y escritura | lectura y escritura |
| Ingesta | lectura de `id_dispositivo`, `dev_eui`, estados | — |

La fila del Configurador es el punto interesante. Dar de alta un dispositivo exige cargar su clave
—`app_key` es `NOT NULL`— y rotarla periódicamente es parte de la operación normal. Pero
**administrar una credencial no requiere poder leerla**. PostgreSQL permite otorgar `INSERT` y
`UPDATE` sobre una columna sin otorgar `SELECT`:

```sql
GRANT INSERT (…, app_key, at_pin, ota_pin, …) ON dispositivo TO rol_configurador;
GRANT UPDATE (…, app_key, at_pin, ota_pin, …) ON dispositivo TO rol_configurador;
-- pero NO: GRANT SELECT (app_key) …
```

El resultado es que el perfil que más manipula dispositivos no puede extraer las claves de ninguno.

Se distingue además entre **credenciales** e **identificadores**: `dev_eui` y `app_eui` viajan en
claro por el aire y no habilitan por sí solos el alta en la red, así que se tratan como datos
comunes. `app_key` es el secreto que autentica al dispositivo, y es el que se protege.

La vista `v_dispositivo` acompaña este corte con un detalle práctico: un `SELECT *` sobre la tabla
falla para el Operador, porque alcanza columnas que no puede leer; sobre la vista funciona.

## 13.5 Autenticación: contraseñas que no viajan ni se guardan en claro

El DDL define `usuario.contrasena TEXT` y el generador de datos inserta el valor tal cual. Para
corregirlo **sin modificar el esquema de B ni su script de carga**, se intercepta la escritura:

```sql
CREATE TRIGGER tg_usuario_hash_contrasena
    BEFORE INSERT OR UPDATE OF contrasena ON usuario
    FOR EACH ROW EXECUTE FUNCTION seguridad.fn_hash_contrasena();
```

La función guarda un hash bcrypt (`crypt()` + `gen_salt('bf', 10)`, de `pgcrypto`) y detecta si el
valor ya venía hasheado, de modo que admite tanto una aplicación que hashea del lado del cliente como
el script de carga que no lo hace, y no vuelve a hashear en cada `UPDATE` de la fila.

La verificación no requiere exponer la tabla:

```sql
SELECT * FROM seguridad.verificar_credenciales('ana@ejemplo.com', '…');
```

Es una función `SECURITY DEFINER`: se ejecuta con los privilegios de su dueño, así que **el login
funciona sin que ningún rol de aplicación tenga `SELECT` sobre `usuario`**. Dos precauciones que
acompañan a toda función `SECURITY DEFINER` y que están aplicadas:

- `SET search_path = public, pg_temp` — sin esto, quien la invoca podría anteponer un esquema propio
  y hacer que la función resuelva a objetos que él controla. Es la vía clásica de escalar privilegios
  a través de una función definer.
- `REVOKE ALL … FROM PUBLIC` antes del `GRANT EXECUTE`: PostgreSQL otorga `EXECUTE` a `PUBLIC` por
  defecto, de modo que un `GRANT` sin el `REVOKE` previo no restringe nada.

**Limitación asumida.** Hashear en el motor implica que la contraseña en claro viaja del cliente al
servidor y puede aparecer en `pg_stat_activity` o en el log de sentencias. Lo correcto en producción
es hashear en la aplicación; el trigger queda como red de contención que garantiza que, venga de
donde venga, en la tabla no quede texto plano.

## 13.6 Auditoría

Un esquema `auditoria` con una única tabla `registro_cambios` y un trigger genérico sobre las **16
tablas de configuración e identidad**. Cuatro decisiones que vale la pena justificar:

**Enmascaramiento de valores sensibles.** El registro guarda la fila anterior y la nueva como `JSONB`,
pero reemplaza por `"***"` el contenido de `contrasena`, `app_key`, `at_pin` y `ota_pin`. Un log que
copia esas columnas concentra en una sola tabla —y en versión histórica— los secretos de todas las
demás, y se convierte en el objetivo más apetecible de la base. Al mismo tiempo, la columna
`columnas_modificadas` registra **que** la credencial cambió sin registrar **a qué** cambió, que es
exactamente lo que una auditoría necesita saber.

**Doble identidad.** Se guardan `session_user` (el rol de base con el que se autenticó la conexión) y
`app.id_usuario`, una variable de sesión que la aplicación declara al tomar una conexión del pool. Con
un pool, todas las sesiones comparten el mismo rol de base: sin el segundo dato, la auditoría sólo
puede decir «lo hizo `app_configurador`», que es tanto como no decir nada.

Un detalle técnico que costó descubrir: dentro de una función `SECURITY DEFINER`, `current_user`
devuelve el **dueño** de la función, no quien la invoca. Registrar `current_user` habría firmado toda
la auditoría como `postgres`. Se registra `session_user`, que sí conserva la identidad de la conexión.

**Inmutabilidad.** Un trigger rechaza `UPDATE` y `DELETE` sobre la tabla de auditoría, incluso para su
dueño. Ningún rol recibe `INSERT`: las filas entran únicamente por los triggers, que corren como
`SECURITY DEFINER`. Un usuario que puede escribir su propia auditoría puede maquillarla.

**Qué NO se audita.** `medicion` y `evento_alarma` quedan fuera a propósito. Son series de sólo
inserción y altísimo volumen: auditarlas duplicaría la tabla más grande del sistema para registrar
que la ingesta hizo lo único que sabe hacer. Su trazabilidad ya está dada por `fecha_hora`,
`contador_mensajes` y el gateway receptor.

La decisión se puede verificar con números: una corrida completa de `popular_tablas()` carga 50.000
mediciones y deja **121 registros de auditoría**. Sin la exclusión serían más de cincuenta mil.

**`TRUNCATE` necesita su propio trigger.** Los triggers de fila no se disparan con `TRUNCATE`:
PostgreSQL vacía la tabla sin recorrerla, así que no hay `OLD` por cada fila que registrar. El efecto
es que la operación más destructiva del catálogo sería justamente la que no deja rastro. No es
hipotético en este proyecto: el procedimiento `popular_tablas()` (`02_populate_tables.sql`, tarea
`C-01`) arranca con un `TRUNCATE … RESTART IDENTITY CASCADE` sobre las 18 tablas, de modo que una
recarga completa de la base pasaría inadvertida.

Se agrega entonces un segundo trigger, a nivel de sentencia, que registra qué tabla se vació, quién y
cuándo. El registro es necesariamente más pobre que el de fila —no guarda el contenido borrado, porque
el motor no llega a leerlo—, pero conserva el dato que importa. La misma corrida de `popular_tablas()`
deja ahora **16 registros de operación `TRUNCATE`** que antes no existían.

## 13.7 Aislamiento por establecimiento

Los perfiles responden *qué* puede hacer un usuario. Falta la otra mitad: *sobre qué filas*. Sin
aislamiento, un operador contratado por un establecimiento consulta las mediciones de otro, y la
plataforma no puede dar servicio a más de un cliente —ni a un contratista que trabaja para varios—
sin exponer los datos de unos a otros.

Se implementa con **Row Level Security** sobre `campo`, `lote`, `sector`, `pivote`, `dispositivo`,
`instalacion_dispositivo`, `medicion` y `evento_alarma`, a partir de una tabla nueva
`seguridad.acceso_campo` (N:M entre usuario y campo).

Cuatro decisiones:

**El default es cerrado.** Si la aplicación no declaró `app.id_usuario`, la función de contexto
devuelve `NULL` y las políticas no dejan ver nada. Olvidarse de fijar el contexto produce una pantalla
vacía, no una fuga. Es la única orientación aceptable para un default en seguridad, y está verificada
como control explícito.

**Los dispositivos sin instalar son visibles para todos.** Un dispositivo se considera visible si su
instalación vigente cae en un campo habilitado; los que todavía no se instalaron pertenecen al stock
común. Sin esa salvedad, un dispositivo recién dado de alta sería invisible para el Configurador que
tiene que instalarlo: el aislamiento volvería imposible la operación que lo precede.

**El dueño de las tablas no está sujeto a RLS**, porque no se declara `FORCE ROW LEVEL SECURITY`. Es
intencional: `db/datos/main.py` carga la base como `postgres` y debe poder escribir en todos los
campos. La contrapartida es que ninguna aplicación debe conectarse con ese rol.

**La resolución se hace en funciones `SECURITY DEFINER`**, no en subconsultas dentro de las políticas.
Además del costo, encadenar políticas entre tablas produce dependencias circulares que terminan en
errores de recursión difíciles de diagnosticar.

**Costo medido.** Aplicar RLS sobre la hipertabla tiene un precio, y conviene darlo en números en vez
de en adjetivos. Midiendo sobre la carga que produce `popular_tablas()` (50.000 mediciones,
PostgreSQL 16), contando las mediciones de los últimos 7 días:

| Escenario | Tiempo de ejecución |
| --- | --- |
| Sin RLS (dueño de la tabla, la saltea) | ~0,2 ms |
| Con RLS, llamada directa a la función de contexto | ~27 ms |
| Con RLS, misma función envuelta en subconsulta escalar | **~7 ms** |

La diferencia entre las dos últimas filas es una sola decisión de escritura de la política, y es
contraintuitiva. Declarar una función `STABLE` **no** garantiza que el motor la evalúe una sola vez:
escrita directamente en la política, la llamada se incorpora al `Filter` del plan y se ejecuta *una
vez por fila*. Como `seguridad.sin_restriccion_de_campo()` hace dos `pg_has_role()`, eso son 100.000
resoluciones de pertenencia a rol para devolver unos cientos de filas. Envolverla en `(SELECT …)` la convierte en
un `InitPlan`, que el planificador evalúa una vez antes de recorrer la tabla.

El plan sigue siendo un recorrido secuencial: el predicado de seguridad se evalúa **antes** que el del
usuario —tiene que ser así, o el filtro del usuario podría revelar filas que no le corresponden— y en
consecuencia el índice sobre `fecha_hora` no llega a aplicarse.

**Una hipótesis que la medición descartó.** La salida evidente ante un costo alto parecía ser
desnormalizar `id_campo` dentro de `medicion` y filtrar por columna propia. Se probó: con la columna agregada, poblada e indexada por `(id_campo, fecha_hora)`,
el tiempo no se movió — ninguna mejora medible. El costo no estaba en resolver a qué
campo pertenece la medición, sino en la llamada por fila. La columna desnormalizada habría agregado un
dato duplicado que mantener a cambio de nada, y por eso **no se incorpora**. Queda anotado porque la
hipótesis descartada es parte del resultado.

Las cifras son de PostgreSQL 16 sobre 50.000 mediciones y varían entre corridas; lo que importa no es
el número exacto sino el orden de magnitud y la relación entre las tres filas.

**Qué queda pendiente de medir.** 50.000 filas es un volumen de validación, no de producción. Con
millones de mediciones el recorrido secuencial deja de ser viable y habrá que evaluar caminos que no
se probaron acá: restringir el alcance temporal desde la aplicación para que Timescale descarte
*chunks* antes de aplicar la política, o mover el filtro por campo a la vista analítica y quitar la RLS
de `medicion`, aceptando que el aislamiento pase a depender de la vista. Es material para §14.

## 13.8 Riesgo de exposición de datos en aplicaciones conectadas a modelos de IA

El trabajo contempla explícitamente futuras aplicaciones de IA sobre estos datos, y eso introduce un
riesgo que no existe en una aplicación tradicional.

**Cuál es el riesgo.** Una aplicación convencional decide qué mostrar en cada pantalla: el programador
escribe la consulta y elige las columnas. Un asistente conversacional o un agente con acceso a la base
no tiene ese filtro — responde con lo que puede leer. Todo lo que su rol alcance es potencialmente
citable en una respuesta, combinable con otros datos, o filtrable mediante una pregunta hábil. A eso
se suman tres vectores propios:

1. **Inyección indirecta de instrucciones.** Los campos de texto libre del sistema —nombre de un
   dispositivo, descripción de una regla de alarma— los escribe un usuario y los lee el modelo. Un
   texto redactado como instrucción puede alterar el comportamiento del agente.
2. **Salida del perímetro.** Prompts, respuestas y trazas suelen enviarse a un proveedor externo. Un
   dato que salió del motor ya no está gobernado por los `GRANT` de este script.
3. **Copias fuera de control.** Todo pipeline que extrae datos para entrenar o para calcular
   *embeddings* crea una copia que **no hereda ni los permisos ni la RLS** de PostgreSQL.

**Qué se hizo al respecto.** La conclusión de diseño es que el control efectivo no es instruir al
modelo para que no revele datos, sino **que el rol con el que se conecta no tenga acceso al dato**.
Las instrucciones se pueden eludir; un privilegio inexistente, no.

- `rol_analitico` no tiene acceso a **ninguna tabla**. Su única puerta es `v_medicion_analitica`.
- Esa vista se construyó a la inversa de las demás: no se parte de la tabla quitando columnas, se
  parte de la pregunta analítica y se agrega sólo lo necesario. Una medición se explica por su
  variable, su ubicación y su momento. Ningún dato de persona entra en la vista, y el dispositivo
  aparece como identificador y tipo, nunca con sus credenciales.
- La vista resuelve la ubicación **histórica** de cada medición, no la actual: un sensor reinstalado
  en otro sector no reescribe el pasado, que es además lo correcto para entrenar cualquier modelo.
- `CONNECTION LIMIT` sobre las identidades de servicio, para acotar el impacto de un cliente que abra
  sesiones sin cerrarlas.
- Recomendación operativa: **ningún agente debe conectarse con `app_administrador`**, ni siquiera para
  «tareas de mantenimiento».

Esta decisión tiene consecuencias en la §11: mantener el eventual índice vectorial *dentro* de
PostgreSQL (`pgvector`) en lugar de en un motor separado permite que las mismas políticas de acceso
alcancen a los *embeddings*. Un almacén vectorial aparte sería, por construcción, una copia de los
datos fuera del perímetro de seguridad.

## 13.9 Verificación

Un `GRANT` escrito no es un permiso verificado. `anexos/verificacion_seguridad.sql` ejercita **25
controles** desde el rol que corresponde en cada caso y reporta si el motor permite o rechaza la
operación:

```
docker exec -i bdia_tp psql -U postgres -d bdia_tp < anexos/verificacion_seguridad.sql
```

Crea sus propios datos de prueba, los usa y termina en `ROLLBACK`, de modo que puede correrse sobre
una base vacía o ya cargada, las veces que haga falta. Al momento de la entrega los 25 controles pasan,
tanto sobre una base recién inicializada como sobre una con los datos de `popular_tablas()` ya cargados
(50.000 mediciones).

El script no es decorativo: **encontró un error real en la primera versión de estos permisos**. Para
evitar tener que escribir un `GRANT` por cada vista futura, `04_roles_permisos.sql` incluía

```sql
ALTER DEFAULT PRIVILEGES IN SCHEMA public GRANT SELECT ON TABLES TO rol_operador;
```

Ese privilegio por defecto alcanzó también a `v_medicion_analitica` —creada más abajo, en el mismo
script— y el Operador terminó con acceso de lectura a la única vista que cruza todos los
establecimientos, anulando en silencio el aislamiento por campo. Ninguna línea del script decía
«otorgar al Operador»; el permiso apareció solo. Es el mismo razonamiento del §13.3 llevado al
extremo: **lo que se abre por defecto se abre también para lo que todavía no existe**. La corrección
fue eliminar el privilegio por defecto sobre tablas y otorgar cada vista a mano.

## 13.10 Limitaciones y trabajo pendiente

Se enumeran explícitamente porque una sección de seguridad que sólo lista lo que funciona es
sospechosa.

| Limitación | Impacto | Camino de solución |
| --- | --- | --- |
| `app_key` se guarda en claro (protegida por permisos, no cifrada) | Un backup o un acceso al filesystem expone las claves | Cifrado a nivel columna con clave custodiada fuera de la base (KMS); `pgcrypto` solo trasladaría el problema a dónde guardar la clave |
| Hash de contraseña en el motor | La contraseña en claro llega al servidor | Hashear en la aplicación; el trigger queda como red de contención |
| Sin cifrado en tránsito | Tráfico legible en la red local | Habilitar TLS en el servidor y `sslmode=require` en los clientes |
| Sin `pgaudit` | Sólo se auditan cambios, no lecturas | Agregar `pgaudit` si se requiere auditar `SELECT` sobre datos sensibles |
| Contraseñas de rol con valores por defecto en el repositorio | Aceptable en local, inaceptable en producción | El script ya lee variables de entorno (`\getenv`); falta conectarlas a un gestor de secretos |
| Sin retención sobre `auditoria.registro_cambios` | Crecimiento indefinido | Particionar por rango de fecha y `DROP` de la partición vencida |
| Costo de RLS sobre la hipertabla medido sólo hasta 50.000 filas | ~15 ms por consulta a ese volumen; el plan es secuencial y no escala linealmente | Volver a medir con volumen de producción; caminos posibles en §13.7 |
| pgAdmin del entorno local con credenciales por defecto | Permite conectarse como superusuario y saltear todo el modelo | Cambiar credenciales y configurar la conexión con un rol de aplicación (§13.2, hallazgo 7) |

---

# 14. Escalabilidad y rendimiento

La solución fue diseñada para escalar en volumen de mediciones IoT y mantener latencia operativa baja en consultas frecuentes.  
Se priorizó una estrategia basada en particionado temporal, retención por horizonte de utilidad y precálculo de agregados.

## 14.1 Estrategia de particionado

La tabla `medicion` se implementa como hipertabla de TimescaleDB particionada por `fecha_hora`, lo que permite:

- distribuir físicamente la serie temporal en fragmentos por tiempo,
- reducir el costo de consultas con filtros temporales,
- sostener tasas de escritura altas sin degradación abrupta.

Este enfoque es consistente con el patrón dominante del caso: inserciones continuas y consultas por ventanas de tiempo (24 h, 7 días, 30 días).

## 14.2 Retención de datos

Se adopta una política por niveles:

- **detalle crudo**: conservar el histórico completo durante el período operativo requerido;
- **horizonte extendido**: para análisis de largo plazo, priorizar agregados por hora/día en lugar de detalle por evento.

La retención evita crecimiento indefinido del almacenamiento de datos crudos y mantiene estable el rendimiento de consulta.

## 14.3 Precálculo y agregaciones

Para consultas repetitivas de monitoreo y tablero, se recomienda precalcular agregados temporales (por ejemplo, promedio de humedad/temperatura por sector y por intervalo).  
Esto reduce latencia en lectura y desacopla el costo de visualización del volumen total histórico.

La estrategia de precálculo complementa los índices: los índices aceleran búsqueda/filtrado; los agregados reducen volumen efectivo a procesar.

## 14.4 Compromisos asumidos

Se asumieron los siguientes compromisos de diseño:

- **flexibilidad vs costo de consulta**: usar JSONB en `valores_medidos` simplifica evolución del esquema, pero exige más costo de CPU en extracción;
- **escritura vs lectura**: se priorizó ingestión continua robusta, complementada con índices para consultas operativas;
- **detalle histórico vs costo de almacenamiento**: mantener todo el detalle mejora trazabilidad, pero requiere política de retención y agregación para escalar.

## 14.5 Evidencia de rendimiento y límites

Las pruebas con `EXPLAIN ANALYZE` muestran mejoras relevantes con índices en consultas operativas clave (sección 10.7), especialmente en consultas con joins y filtros temporales.  
La mejora no es homogénea entre consultas: cuando predomina ordenamiento/ventana, la ganancia es menor que en consultas fuertemente filtradas por tiempo.